# Lecture 6 self-check: introduction to small language models

Twelve short questions on the main points of today's lecture, in the order the lecture takes them. The quiz is not assessed: it is a check that you have the main ideas before you start this afternoon's problem set. Each question gives everything you need to answer it, and the code in it runs in the module's `mpm2026` environment. The answers, with explanations, will be released later so that you can check your own.

Run the code cell below first (or use *Run All*). Then click one option under each question. Your choices are listed at the end of the notebook. They are kept until you restart the kernel, so note them down if you want to keep them.

In [ ]:
# Run this cell first. It sets up the clickable options under each question.
from IPython.display import display
from ipywidgets import Layout, RadioButtons

my_answers = {}


def choose(question, letters):
    """Show clickable options for one question and remember your choice."""
    buttons = RadioButtons(options=list(letters), value=None, layout=Layout(width="auto"))
    buttons.observe(lambda change: my_answers.__setitem__(question, change["new"]), names="value")
    display(buttons)

### Question 1

The lecture's `bigram_counts` is applied to the text `"abracadabra"`, and the counts are normalised without smoothing:

```python
import numpy as np

def bigram_counts(ids, V):
    """Count adjacent pairs. N[i, j] = number of times j followed i."""
    N = np.zeros((V, V), dtype=np.int64)
    np.add.at(N, (ids[:-1], ids[1:]), 1)
    return N


text = "abracadabra"
chars = sorted(set(text))                    # ['a', 'b', 'c', 'd', 'r']
stoi = {c: i for i, c in enumerate(chars)}
ids = np.array([stoi[c] for c in text])

N = bigram_counts(ids, len(chars))
P = N / N.sum(axis=1, keepdims=True)
```

What is `P[stoi["a"], stoi["b"]]`?

- **(A)** `0.2`
- **(B)** `0.4`
- **(C)** `0.5`
- **(D)** `1.0`


In [ ]:
choose(1, "ABCD")

### Question 2

The lecture's numerically stable softmax is

```python
import numpy as np

def softmax(z):
    """Numerically stable softmax along the last axis."""
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)
```

What does `softmax(np.array([1000.0, 1000.0, 1001.0]))` return, to two decimal places?

- **(A)** `[0.21, 0.21, 0.58]`
- **(B)** `[0.00, 0.00, 1.00]`
- **(C)** `[0.25, 0.25, 0.50]`
- **(D)** `[0.33, 0.33, 0.33]`


In [ ]:
choose(2, "ABCD")

### Question 3

In the lecture's `sample_next`, the model's probabilities become logits with `np.log`, the logits are divided by the temperature, and the next token is drawn from the softmax of the result:

```python
logits = np.log(probs + 1e-12) / temperature
...
return rng.choice(len(probs), p=softmax(logits))
```

With `probs = np.array([0.6, 0.3, 0.1])` and `temperature=0.5` (and no top-k), what distribution is the next token drawn from, to two decimal places? (Rounded values may not add up to exactly 1.)

- **(A)** `[1.00, 0.00, 0.00]`
- **(B)** `[0.60, 0.30, 0.10]`
- **(C)** `[0.47, 0.33, 0.19]`
- **(D)** `[0.78, 0.20, 0.02]`


In [ ]:
choose(3, "ABCD")

### Question 4

This is how the lecture's `sample_next` applies top-k:

```python
logits = np.log(probs + 1e-12) / temperature

if top_k is not None:
    threshold = np.sort(logits)[-top_k]
    logits = np.where(logits < threshold, -np.inf, logits)

return rng.choice(len(probs), p=softmax(logits))
```

With `probs = np.array([0.1, 0.4, 0.2, 0.3])`, `temperature=1.0` and `top_k=2`, what distribution is the next token drawn from, to two decimal places? (Rounded values may not add up to exactly 1.)

- **(A)** `[0.00, 0.52, 0.00, 0.48]`
- **(B)** `[0.00, 0.57, 0.00, 0.43]`
- **(C)** `[0.00, 0.55, 0.00, 0.45]`
- **(D)** `[0.00, 0.50, 0.00, 0.50]`


In [ ]:
choose(4, "ABCD")

### Question 5

A bigram model over three characters (character ids 0, 1 and 2), a held-out text of four characters, `ids`, and the lecture's `nll` (Exercise 6.2):

```python
import numpy as np

P = np.array([[0.25,  0.5,  0.25 ],     # P[i, j] = P(next = j | current = i)
              [0.5,   0.25, 0.25 ],
              [0.125, 0.75, 0.125]])
ids = np.array([0, 1, 2, 0])

def nll(P, ids):
    """Mean negative log-likelihood per character of `ids` under bigram model P."""
    return -np.log(P[ids[:-1], ids[1:]]).mean()
```

What is the model's perplexity on `ids`, to two decimal places?

- **(A)** `64.00`
- **(B)** `4.67`
- **(C)** `1.39`
- **(D)** `4.00`


In [ ]:
choose(5, "ABCD")

### Question 6

A bigram model has a vocabulary of four characters: `a`, `b`, `c` and `d`. In the training text, `a` was followed 6 times by `b` and twice by `d`, and never by `a` or `c`. With add-1 smoothing, which pretends that every possible bigram was seen once more than it was, what probability does the model give to `c` coming next after `a`?

- **(A)** 1/4
- **(B)** 1/8
- **(C)** 1/9
- **(D)** 1/12


In [ ]:
choose(6, "ABCD")

### Question 7

An order-5 count model predicts each character from the previous five, using the counts in its training text with add-1 smoothing (as `ngram_nll` does), over a vocabulary of V = 65 characters. In the training text the context `shalt` occurs a dozen times, but the context `shalb` never occurs. What distribution does the model use for the character after `shalb`?

- **(A)** Probability 1/65 for every character: each is counted once, by the smoothing alone
- **(B)** The add-1 smoothed distribution after `b`: the model falls back on the last character of the context
- **(C)** Nearly the distribution after `shalt`: the two contexts differ only in their last character
- **(D)** `nan` for every character: the context's counts are all zero, and normalising them divides 0 by 0


In [ ]:
choose(7, "ABCD")

### Question 8

The lecture's `train_bpe` starts from a text's bytes, counts the adjacent pairs, merges the most frequent pair into a new token, and repeats. With

```python
merges, vocab, ids = train_bpe("he the he then", n_merges=2)
```

the first new token, `vocab[256]`, is `b'he'`. What is the second, `vocab[257]`?

- **(A)** `b'e '`
- **(B)** `b' t'`
- **(C)** `b'he '`
- **(D)** `b'the'`


In [ ]:
choose(8, "ABCD")

### Question 9

In Part 6(a) a bigram model is learned instead of counted: a V × V matrix `W` starts at zero, `softmax(W[x])` is the model's distribution for the character after `x`, and gradient descent lowers the mean negative log-likelihood of the training pairs. Suppose it is run without the regularisation term, using every training pair at every step, until the loss stops changing, and that each of the V² possible pairs occurs at least once in the training text. What is `softmax(W[x])` then, for each character `x`?

- **(A)** The same for every `x`: the fraction of the training text that each character makes up
- **(B)** The counts of the characters that followed `x`, divided by their total
- **(C)** Nearly all the probability on the character that most often followed `x`
- **(D)** The counts of the characters that followed `x`, each plus 1, divided by their total


In [ ]:
choose(9, "ABCD")

### Question 10

The lecture's single-head self-attention is

```python
import numpy as np

def attention(X, Wq, Wk, Wv, causal=True):
    """Single-head self-attention."""
    Q, K, Vv = X @ Wq, X @ Wk, X @ Wv

    scores = Q @ K.T / np.sqrt(Wk.shape[1])

    if causal:
        T = X.shape[0]
        scores = np.where(np.tril(np.ones((T, T))) == 0, -np.inf, scores)

    A = softmax(scores)
    return A @ Vv, A
```

where `softmax` is the lecture's numerically stable softmax, applied to each row. It is run on `T = 5` positions with the query and key projections set to zero:

```python
T, d, d_k = 5, 6, 4
X = np.random.default_rng(0).normal(size=(T, d))
out, A = attention(X, np.zeros((d, d_k)), np.zeros((d, d_k)), np.eye(d))
```

What is `A[2]`, the attention weights of position 2?

- **(A)** `[1/3, 1/3, 1/3, 0, 0]`
- **(B)** `[1/5, 1/5, 1/5, 1/5, 1/5]`
- **(C)** `[1/2, 1/2, 0, 0, 0]`
- **(D)** `[0, 0, 1, 0, 0]`


In [ ]:
choose(10, "ABCD")

### Question 11

You tell a coding assistant that you think the bug in your code is in NumPy, and it agrees straight away, although the bug turns out to be in your own code. The lecture describes three stages of training: pretraining, instruction tuning and preference tuning. Which of them does it say gives a model this tendency to agree with you?

- **(A)** Pretraining: next-token prediction on an enormous corpus of text
- **(B)** Preference tuning: training against human ratings of which of two responses is better
- **(C)** Instruction tuning: training on examples of instructions and good responses to them


In [ ]:
choose(11, "ABC")

### Question 12

You ask an AI assistant for a faster version of a slow function, pasting in only the function. You also work out what it hasn't been given: it hasn't seen the test the function has to pass, the size of your real input, or the rest of your package. Which of SCOPE's five steps (State, Characterise, Obtain, Probe, Evaluate) is working out what the assistant hasn't been given?

- **(A)** Evaluate and record
- **(B)** Probe the output
- **(C)** Characterise the operator
- **(D)** State the task


In [ ]:
choose(12, "ABCD")

## Your answers

In [ ]:
for q in range(1, 13):
    print(f"Question {q}: {my_answers.get(q) or 'not answered'}")